# 🦙 Local Ollama Connection Demo — `qwen3:4b`

This notebook demonstrates **two methods** to connect to a locally running Ollama instance using the `qwen3:4b` model:

| Method | Library | Description |
|--------|---------|-------------|
| **Method 1** | `ollama` (official Python SDK) | Clean, idiomatic Python SDK |
| **Method 2** | `requests` (REST API) | Raw HTTP calls to Ollama's REST API |

> ✅ Make sure Ollama is running locally before executing cells: `ollama serve`

---
## 📦 Step 0 — Install Dependencies

In [1]:
# Run this cell once to install all required packages
!pip install ollama requests ipywidgets

---
## ⚙️ Step 1 — Configuration

In [2]:
# ─────────────────────────────────────────────
#  Shared Configuration
# ─────────────────────────────────────────────

MODEL_NAME   = "qwen3:4b"           # Ollama model tag
OLLAMA_HOST  = "http://localhost:11434"  # Default Ollama server address

SYSTEM_PROMPT = """
You are a helpful, concise, and friendly AI assistant.
Always respond in clear English.
If you are unsure about something, say so honestly.
Keep answers short and to the point unless asked for detail.
""".strip()

USER_PROMPT = "Explain what a Large Language Model is in 3 sentences."

print(f"Model     : {MODEL_NAME}")
print(f"Host      : {OLLAMA_HOST}")
print(f"System    : {SYSTEM_PROMPT[:60]}...")
print(f"User Msg  : {USER_PROMPT}")

Model     : qwen3:4b
Host      : http://localhost:11434
System    : You are a helpful, concise, and friendly AI assistant.
Alway...
User Msg  : Explain what a Large Language Model is in 3 sentences.


---
## 🔍 Step 2 — Health Check (Verify Ollama is Running)

In [3]:
import requests

def check_ollama_health(host: str = OLLAMA_HOST) -> bool:
    """Ping the Ollama server and confirm it's reachable."""
    try:
        resp = requests.get(f"{host}/api/tags", timeout=5)
        if resp.status_code == 200:
            models = [m["name"] for m in resp.json().get("models", [])]
            print(f"✅ Ollama is running!")
            print(f"   Available models: {models if models else '(none pulled yet)'}")
            if MODEL_NAME not in models and not any(MODEL_NAME in m for m in models):
                print(f"⚠️  '{MODEL_NAME}' not found. Run: ollama pull {MODEL_NAME}")
            return True
    except requests.exceptions.ConnectionError:
        print(f"❌ Cannot reach Ollama at {host}")
        print("   → Start it with: ollama serve")
    return False

check_ollama_health()

✅ Ollama is running!
   Available models: ['qwen3:4b', 'gemma3:4b']


True

---
## 🟢 Method 1 — Official `ollama` Python SDK

The `ollama` package is the **recommended** way to interact with a local Ollama instance.  
It provides a clean, typed interface with both sync and async support.

In [4]:
import ollama

# ── 1-A  Basic Chat (Non-Streaming) ──────────────────────────────────────────
print("=" * 60)
print(" METHOD 1-A │ ollama SDK — Basic Chat")
print("=" * 60)

response = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {"role": "system",  "content": SYSTEM_PROMPT},
        {"role": "user",    "content": USER_PROMPT},
    ]
)

answer = response["message"]["content"]

print(f"\n📤 User   : {USER_PROMPT}")
print(f"\n🤖 Model  : {answer}")
print(f"\n📊 Tokens — prompt: {response['prompt_eval_count']} | "
      f"completion: {response['eval_count']}")

 METHOD 1-A │ ollama SDK — Basic Chat

📤 User   : Explain what a Large Language Model is in 3 sentences.

🤖 Model  : A Large Language Model (LLM) is a type of artificial intelligence that learns patterns from vast amounts of text data to understand and generate human-like language. It trains on enormous datasets to predict the next word in a sequence, building knowledge across topics and contexts. This lets it answer questions, write stories, code, and more—really making it a powerful tool for many real-world tasks.

📊 Tokens — prompt: 69 | completion: 367


In [5]:
import ollama
import sys

# ── 1-B  Streaming Chat (token-by-token output) ───────────────────────────────
print("=" * 60)
print(" METHOD 1-B │ ollama SDK — Streaming")
print("=" * 60)

STREAM_PROMPT = "Write a short Python function that checks if a number is prime."

print(f"\n📤 User   : {STREAM_PROMPT}")
print(f"\n🤖 Model  : ", end="", flush=True)

stream = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": STREAM_PROMPT},
    ],
    stream=True,
)

for chunk in stream:
    token = chunk["message"]["content"]
    print(token, end="", flush=True)

print("\n")  # newline after stream ends

 METHOD 1-B │ ollama SDK — Streaming

📤 User   : Write a short Python function that checks if a number is prime.

🤖 Model  : Here's a concise prime-checking function:

```python
def is_prime(n):
    return n >= 2 and all(n % i != 0 for i in range(2, int(n**0.5) + 1))
```



In [ ]:
import ollama

# ── 1-C  Multi-Turn Conversation ─────────────────────────────────────────────
print("=" * 60)
print(" METHOD 1-C │ ollama SDK — Multi-Turn Conversation")
print("=" * 60)

conversation_history = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

turns = [
    "What is the capital of France?",
    "What is its population approximately?",
    "Name one famous landmark there.",
]

for user_msg in turns:
    conversation_history.append({"role": "user", "content": user_msg})

    resp = ollama.chat(model=MODEL_NAME, messages=conversation_history)
    bot_reply = resp["message"]["content"]

    conversation_history.append({"role": "assistant", "content": bot_reply})

    print(f"\n👤 User : {user_msg}")
    print(f"🤖 Bot  : {bot_reply}")

---
## 🔵 Method 2 — Raw REST API via `requests`

Ollama exposes a simple HTTP REST API at `http://localhost:11434`.  
This method requires **no special SDK** — just the standard `requests` library.

In [ ]:
import requests
import json

# ── 2-A  /api/chat endpoint (OpenAI-compatible style) ────────────────────────
print("=" * 60)
print(" METHOD 2-A │ REST API — /api/chat (non-streaming)")
print("=" * 60)

def ollama_chat_rest(
    user_message: str,
    system_prompt: str = SYSTEM_PROMPT,
    model: str = MODEL_NAME,
    host: str = OLLAMA_HOST,
) -> dict:
    """Send a chat request to Ollama's /api/chat endpoint."""
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        "stream": False,   # get a single JSON response
    }
    resp = requests.post(f"{host}/api/chat", json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()


REST_PROMPT = "Give me 3 tips for writing clean Python code."
result = ollama_chat_rest(REST_PROMPT)

print(f"\n📤 User   : {REST_PROMPT}")
print(f"\n🤖 Model  : {result['message']['content']}")
print(f"\n📊 Tokens — prompt: {result.get('prompt_eval_count','?')} | "
      f"completion: {result.get('eval_count','?')}")

In [ ]:
import requests
import json
import sys

# ── 2-B  /api/generate endpoint (raw completion) ─────────────────────────────
print("=" * 60)
print(" METHOD 2-B │ REST API — /api/generate (raw prompt)")
print("=" * 60)

def ollama_generate_rest(
    prompt: str,
    system_prompt: str = SYSTEM_PROMPT,
    model: str = MODEL_NAME,
    host: str = OLLAMA_HOST,
    stream: bool = False,
):
    """Call Ollama's /api/generate endpoint (raw text completion)."""
    payload = {
        "model":  model,
        "system": system_prompt,
        "prompt": prompt,
        "stream": stream,
    }
    resp = requests.post(
        f"{host}/api/generate", json=payload,
        stream=stream, timeout=120
    )
    resp.raise_for_status()
    return resp


GEN_PROMPT = "What are the main differences between Python lists and tuples?"
print(f"\n📤 User   : {GEN_PROMPT}")
print(f"\n🤖 Model  : ", end="", flush=True)

# Stream the response token by token
resp = ollama_generate_rest(GEN_PROMPT, stream=True)
for line in resp.iter_lines():
    if line:
        chunk = json.loads(line)
        print(chunk.get("response", ""), end="", flush=True)
        if chunk.get("done"):
            break
print()

In [ ]:
import requests
import json

# ── 2-C  OpenAI-Compatible endpoint (/v1/chat/completions) ───────────────────
print("=" * 60)
print(" METHOD 2-C │ REST API — OpenAI-Compatible /v1/chat/completions")
print("=" * 60)

def ollama_openai_compat(
    user_message: str,
    system_prompt: str = SYSTEM_PROMPT,
    model: str = MODEL_NAME,
    host: str = OLLAMA_HOST,
) -> str:
    """Use Ollama's OpenAI-compatible API (drop-in for openai SDK)."""
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
    }
    headers = {"Content-Type": "application/json"}
    resp = requests.post(
        f"{host}/v1/chat/completions",
        headers=headers, json=payload, timeout=120
    )
    resp.raise_for_status()
    data = resp.json()
    return data["choices"][0]["message"]["content"]


OAI_PROMPT = "Summarize the concept of recursion in programming."
answer = ollama_openai_compat(OAI_PROMPT)

print(f"\n📤 User   : {OAI_PROMPT}")
print(f"\n🤖 Model  : {answer}")

---
## 📊 Step 3 — Side-by-Side Comparison

In [ ]:
import ollama, requests, time

COMPARE_PROMPT = "What is machine learning? Answer in exactly 2 sentences."
results = {}

# ── Method 1: ollama SDK ─────────────────────────────────────────────────────
t0 = time.time()
r1 = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": COMPARE_PROMPT},
    ]
)
results["ollama SDK"] = {
    "answer": r1["message"]["content"],
    "elapsed": round(time.time() - t0, 2),
    "tokens": r1.get("eval_count", "?"),
}

# ── Method 2: REST API ───────────────────────────────────────────────────────
t0 = time.time()
r2 = requests.post(
    f"{OLLAMA_HOST}/api/chat",
    json={
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": COMPARE_PROMPT},
        ],
        "stream": False,
    },
    timeout=120,
).json()
results["REST API"] = {
    "answer": r2["message"]["content"],
    "elapsed": round(time.time() - t0, 2),
    "tokens": r2.get("eval_count", "?"),
}

# ── Print comparison ─────────────────────────────────────────────────────────
print(f"Prompt: {COMPARE_PROMPT}\n")
print(f"{'─'*60}")
for method, data in results.items():
    print(f"\n🔧 {method}")
    print(f"   ⏱  Time   : {data['elapsed']}s")
    print(f"   📊 Tokens : {data['tokens']}")
    print(f"   💬 Answer : {data['answer']}")
    print(f"{'─'*60}")

---
## 🎯 Step 4 — Interactive Prompt (Bonus)

A simple text widget to chat with the model directly in the notebook.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import ollama

system_input = widgets.Textarea(
    value=SYSTEM_PROMPT,
    description="System:",
    layout=widgets.Layout(width="98%", height="80px"),
)
user_input = widgets.Text(
    placeholder="Type your message here...",
    description="You:",
    layout=widgets.Layout(width="80%"),
)
send_btn  = widgets.Button(description="Send", button_style="primary")
clear_btn = widgets.Button(description="Clear", button_style="warning")
out       = widgets.Output()

chat_history = []

def on_send(b):
    msg = user_input.value.strip()
    if not msg:
        return
    chat_history.append({"role": "user", "content": msg})
    user_input.value = ""

    messages = [{"role": "system", "content": system_input.value}] + chat_history
    resp = ollama.chat(model=MODEL_NAME, messages=messages)
    reply = resp["message"]["content"]
    chat_history.append({"role": "assistant", "content": reply})

    with out:
        clear_output(wait=True)
        for m in chat_history:
            icon = "👤" if m["role"] == "user" else "🤖"
            print(f"{icon} {m['role'].upper()}: {m['content']}\n")

def on_clear(b):
    chat_history.clear()
    with out:
        clear_output()

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)

display(
    system_input,
    widgets.HBox([user_input, send_btn, clear_btn]),
    out
)

---
## ✅ Summary

| Feature | Method 1: `ollama` SDK | Method 2: `requests` REST |
|---------|----------------------|---------------------------|
| Ease of use | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ |
| Extra dependencies | `ollama` package | none (stdlib) |
| Streaming support | ✅ built-in | ✅ manual |
| OpenAI-compatible | ✅ | ✅ `/v1/chat/completions` |
| Best for | Most use-cases | Portability / no-install |
